In [6]:
import os

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from bluemath_tk.wrappers.swash.swash_wrapper import ChySwashModelWrapper
from bluemath_tk.datamining import MDA

In [7]:
#### Load sea states data

sea_states = pd.read_csv('data/sea_states.txt',sep='\t')
sea_states.columns = ['Hs', 'Hs_L0', 'WL']

In [ ]:
mda_ob = MDA(num_centers=10)
mda_ob.fit(data=sea_states)
sea_states_cases = mda_ob.centroids

In [18]:
sea_states_cases['Cf'] = 0.1555
sea_states_cases['Cr'] = 0.7

#### Inputs

In [19]:
outputs_dir = "outputs/hyswash_molokai_dynamic"
if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir)

templates_dir = "inputs/templates/hyswash"
swash_runs_dir = "outputs/swash_molokai_dynamic_cases"

In [20]:
fixed_parameters = {
    "dxinp": 1.5,  # bathymetry grid spacing
    "default_Cf": 0.002,  # Friction manning coefficient (m^-1/3 s)
    "Cf_ini": 700 / 1.5,  # Friction start cell
    "Cf_fin": 1250 / 1.5,  # Friction end cell
    "comptime": 7200,  # Simulation duration (s)
    "warmup": 7200 * 0.15,  # Warmup duration (s)
    "n_nodes_per_wavelength": 60,  # number of nodes per wavelength
}
metamodel_parameters = sea_states_cases.to_dict(orient="list")

swash_wrapper = ChySwashModelWrapper(
    templates_dir=templates_dir,
    metamodel_parameters=metamodel_parameters,
    fixed_parameters=fixed_parameters,
    output_dir=swash_runs_dir,
    depth_array=np.loadtxt(os.path.join(templates_dir, "depth.bot")),
)
swash_wrapper

2026-03-27 04:45:29,095 - ChySwashModelWrapper - WARNING - Parameter Cf is not in the default_parameters
2026-03-27 04:45:29,095 - ChySwashModelWrapper - WARNING - Parameter Cr is not in the default_parameters


In [21]:
swash_wrapper.build_cases(mode="one_by_one", num_workers=10)

In [24]:
swash_wrapper.save_model(model_path=os.path.join(outputs_dir, "swash_model.pkl"), exclude_attributes=['_env'])